# فراسطح تنظیم‌پذیر: از ولتاژ تا پاسخ کنترل‌شده

[فصل](../chapters/07-active-tunable.md)، [راهنمای مثال‌ها](../chapters/07-tunable-guide.md) و [آزمایشگاه](../labs/07-tunable-lab.md) را بخوانید. هسته را از نو آغاز و همهٔ سلول‌ها را اجرا کنید. مدل‌ها شامل مدار معادل غیرفعال RLC تک‌درگاه، دهانهٔ اسکالر تجویزی، بازخورد مصنوعی و پوش مدولاسیون آهسته هستند. هندسهٔ وصله حل یا اندازه‌گیری نشده است. [شماتیک مدل](../assets/chapter-07/tunable-cell.svg) نقش هر تقریب را مشخص می‌کند.

## ۱. پارامتر و بازتولیدپذیری

قرارداد $e^{-i\omega t}$ است. مرجع، ۱۰ گیگاهرتز، تابش عمودی، بدون درگاه عبور و امپدانس مرجع حقیقی مثبت هواست. مقادیر مدار مؤثر، پارامتر آموزشی‌اند؛ داده‌برگ دیود یا استخراج از شکل نیستند. آرایه ۳۲ عنصر با فاصلهٔ نصف طول موجِ ۱۰ گیگاهرتز دارد. تغییر پارامتر مدار، آزمایش تازه و نیازمند کنترل مجدد است.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import hashlib, json, platform, sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd()/'simulations').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from simulations.tunable_surface import *
P=Surface()
FREQUENCY_GHZ=10.
TARGET_DEG=25.
VOLTAGE=np.linspace(0,P.voltage_max_V,401)
FREQUENCIES=np.linspace(7.,13.,241)
ANGLES=np.linspace(-80.,80.,641)
plt.rcParams.update({'figure.dpi':115,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
print('Python',platform.python_version(),'NumPy',np.__version__)
print(json.dumps(asdict(P),indent=2))
print('Equivalent circuit + scalar array + synthetic feedback; no full-wave or measured data.')

## ۲. مسیر پاسخ مختلط دسترس‌پذیر

**پیش‌بینی:** افزایش ولتاژ، فرکانس تشدید را بالا می‌برد یا پایین؟ آیا ۱۶ ولتاژ هم‌فاصله به معنی ۱۶ فاز هم‌فاصله است؟

مدل از $C(V)=C_0/(1+V/V_j)^M$، $Z_s=R+i(1/\omega C-\omega L)$ و $\Gamma=(Z_s-Z_0)/(Z_s+Z_0)$ استفاده می‌کند. سهم اتلاف مستقل $A=4RZ_0/|Z_s+Z_0|^2$ است. در بازتاب صفر، فاز تعریف نمی‌شود؛ گشودن فاز از میان صفر را دامنهٔ تنظیم مفید ننامید. مرجع $R=25\,\Omega$ از این تکینگی دور است.

In [ ]:
sweep=response(FREQUENCY_GHZ,VOLTAGE,P)
phase=np.rad2deg(np.unwrap(np.angle(sweep['gamma'])))
spectral=response(FREQUENCIES[:,None],VOLTAGE[None,:],P)
assert np.max(abs(spectral['reflected']+spectral['absorbed']-1))<2e-15
resonance_GHz=1/(2*np.pi*np.sqrt(P.inductance_nH*1e-9*capacitance(VOLTAGE,P)))/1e9
fig,axes=plt.subplots(1,3,figsize=(13,4.3),constrained_layout=True)
unit=np.exp(1j*np.linspace(0,2*np.pi,400))
axes[0].plot(unit.real,unit.imag,':',color='#abb8c2')
sc=axes[0].scatter(sweep['gamma'].real,sweep['gamma'].imag,c=VOLTAGE,s=9,cmap='viridis')
axes[0].set(xlabel='Re Γ',ylabel='Im Γ',title='Reachable reflection states',aspect='equal',xlim=(-1.1,1.1),ylim=(-1.1,1.1))
fig.colorbar(sc,ax=axes[0],label='Bias V (V)',shrink=.7)
axes[1].plot(VOLTAGE,phase,color='#087f83');axes[1].set(xlabel='Bias V (V)',ylabel='Unwrapped phase (degrees)',title='Phase and reflected fraction')
power_axis=axes[1].twinx();power_axis.plot(VOLTAGE,sweep['reflected'],color='#b36b10',ls='--');power_axis.set(ylabel='|Γ|²',ylim=(0,1.05));power_axis.spines['right'].set_visible(True)
im=axes[2].pcolormesh(VOLTAGE,FREQUENCIES,spectral['absorbed'],shading='auto',cmap='magma')
axes[2].plot(VOLTAGE,resonance_GHz,'w--',lw=1.4,label='LC resonance');axes[2].set(xlabel='Bias V (V)',ylabel='Frequency (GHz)',ylim=(7,13),title='Absorption A = 1 − |Γ|²');axes[2].legend(fontsize=8)
fig.colorbar(im,ax=axes[2],label='Absorbed fraction',shrink=.7)
plt.show()
print(f'Continuous phase span at 10 GHz: {abs(phase[-1]-phase[0]):.3f} degrees')
print(f'Minimum sampled reflection fraction: {sweep["reflected"].min():.6f}')

## ۳. برنامه‌ریزی دهانهٔ اسکالر متناهی

با موقعیت‌های $x_j/\lambda_0$، هدف‌ها $d_j=e^{i2\pi(x_j/\lambda_0)\sin\theta_t}$ هستند. در هر جایگاه حالتی را برگزینید که $|\Gamma_b-d_j|^2$ را کمینه کند. این قانون انتخاب محلی است، نه شکل‌دهی باریکهٔ بهینهٔ سراسری. میدان $F(\theta)=N^{-1}\sum_j\Gamma_j e^{-i2\pi(x_j/\lambda_0)\sin\theta}$ است.

حالت‌های فاز ایده‌آل را با ۱۶ ولتاژ هم‌فاصلهٔ مدار مقایسه کنید. بهنجارسازی مشترک را نگه دارید: $I=|F|^2$ شدت نسبت به مرجع همدوس ایده‌آلِ $N$ منبع است؛ **بازده کل یا جهت‌مندی نیست**. هر منحنی را به قلهٔ خودش بهنجار نکنید. تخمین سینکِ خطای فاز، توزیع یکنواخت خطا را فرض می‌کند؛ رمپ متناهی می‌تواند متفاوت باشد. کدگذاری یک‌بیتی ممکن است باریکهٔ دوم هم‌قدرت بسازد.

In [ ]:
x=array_positions()
ideal=np.exp(2j*np.pi*x*np.sin(np.deg2rad(TARGET_DEG)))
designs={str(b):code_array(TARGET_DEG,b) for b in range(1,5)}
actual=code_array(TARGET_DEG,4,circuit=True)
fig=plt.figure(figsize=(12.5,6.3),layout='constrained');gs=fig.add_gridspec(2,2,width_ratios=[1.4,1])
ax=fig.add_subplot(gs[:,0])
for label,g,color in [('Continuous phase',ideal,'#173044'),('Ideal 1 bit',designs['1']['gamma'],'#a077b4'),('Ideal 2 bit',designs['2']['gamma'],'#b36b10'),('Circuit, 16 voltages',actual['gamma'],'#087f83')]:
    ax.plot(ANGLES,10*np.log10(np.maximum(abs(array_field(g,ANGLES,x))**2,1e-6)),label=label,color=color)
ax.axvline(TARGET_DEG,color='#7c8993',ls=':',lw=1);ax.set(xlabel='Signed output angle (degrees)',ylabel='10 log10 |F|² (dB)',ylim=(-45,1),xlim=(-80,80),title='32 prescribed sources; common normalization');ax.legend(fontsize=9)
ax=fig.add_subplot(gs[0,1]);ax.bar(x,actual['voltage_V'],width=.42,color='#087f83');ax.set(xlabel='Position x / λ₀',ylabel='Bias (V)',title='Selected control pattern')
ax=fig.add_subplot(gs[1,1]);vals=[designs[str(b)]['target_metric'] for b in range(1,5)]+[actual['target_metric']]
ax.bar(['1 bit','2 bit','3 bit','4 bit','Circuit\n4 bit'],vals,color=['#a6b6c4']*4+['#087f83']);ax.set(ylabel='|F(target)|²',ylim=(0,1.1),title='Target-direction metric, not efficiency')
for i,v in enumerate(vals):ax.text(i,v+.025,f'{v:.3f}',ha='center',fontsize=9)
plt.show()
print({k:round(v['target_metric'],6) for k,v in designs.items()})
print('Circuit target metric:',actual['target_metric'])

## ۴. بستن حلقه با پاسخ مختلط اندازه‌گیری‌شده

در سامانهٔ مصنوعی، $C_0$ نسبت به مقدار اسمی ۱۲٪ بیشتر است. کنترل‌گر این پارامتر را دریافت نمی‌کند: بازتاب مختلط را می‌پرسد، ۱۷ ولتاژ را جاروب و بهترین بازه را پالایش می‌کند. هدف، پاسخ اسمی در ۸ ولت است. همهٔ پرس‌وجوها، حتی نتایج ضعیف، حفظ می‌شوند. مثال بدون نویز، سنجش و تنظیم را نشان می‌دهد؛ مقاومت در برابر نویز حسگر، پسماند، کوپلینگ یا بار متغیر را ثابت نمی‌کند.

**کنترل مستقل:** برای $M=1$، برابری ظرفیت می‌دهد $1+V_\mathrm{corrected}/V_j=1.12(1+8/V_j)$؛ پس $V_\mathrm{corrected}=9.08$ ولت. نتیجهٔ جبری را با کنترل‌گر مقایسه کنید.

In [ ]:
plant=replace(P,capacitance0_pF=P.capacitance0_pF*1.12)
target=complex(response(10.,8.,P)['gamma'])
feedback=feedback_voltage(target,lambda v:response(10.,v,plant)['gamma'])
chosen=feedback['selected']['voltage_V'];before=complex(response(10.,8.,plant)['gamma']);after=complex(response(10.,chosen,plant)['gamma'])
assert abs(chosen-9.08)<1e-5
fig,axes=plt.subplots(1,2,figsize=(11.8,4.2),constrained_layout=True)
plant_curve=response(10.,VOLTAGE,plant)['gamma']
axes[0].plot(plant_curve.real,plant_curve.imag,color='#a6b6c4',label='Synthetic plant curve')
for value,label,color,marker in [(target,'Target','#173044','*'),(before,'Nominal 8 V on plant','#b36b10','o'),(after,'After feedback','#087f83','x')]:axes[0].scatter(value.real,value.imag,label=label,color=color,s=75,marker=marker)
axes[0].set(xlabel='Re Γ',ylabel='Im Γ',aspect='equal',title='Calibration in the complex plane');axes[0].legend(fontsize=9)
costs=np.array([q['error_squared'] for q in feedback['queries']])
axes[1].semilogy(np.arange(1,len(costs)+1),np.maximum(costs,1e-20),'.',color='#a6b6c4',label='Every query')
axes[1].semilogy(np.arange(1,len(costs)+1),np.maximum(np.minimum.accumulate(costs),1e-20),color='#087f83',label='Best so far')
axes[1].set(xlabel='Measurement query',ylabel='|Γmeasured − Γtarget|²',title='Coarse scan, then bounded refinement');axes[1].legend()
plt.show()
print(f'Before: complex error {abs(before-target):.6f}; after: {abs(after-target):.3e}')
print(f'Corrected bias: {chosen:.8f} V; {len(costs)} noiseless response queries')

## ۵. فرمان متغیر، حالت الکترومغناطیسی آنی نیست

ولتاژ از $\tau\dot V+V=V_\mathrm{command}$ با $\tau=4\,\mu\mathrm s$ پیروی می‌کند. نشست پله تا ۲٪ تغییر ولتاژ پس از $-\tau\ln(0.02)$ رخ می‌دهد. بازتاب تابع غیرخطی ولتاژ است؛ پس فاز همان نمایی را دنبال نمی‌کند.

برای فرمان تناوبی ۲۰ کیلوهرتز و چرخهٔ کاری ۵۰٪، حالت‌های دودویی آنی را با حالت ماندگار تناوبی RC مقایسه کنید. بسط $\Gamma(t)=\sum_\ell c_\ell e^{-i\ell\Omega t}$، فرکانس‌های بازتابی $\omega_0+\ell\Omega$ را می‌دهد. برای موج دودویی، $c_\ell=(\Gamma_a-\Gamma_b)D\operatorname{sinc}(\ell D)e^{i\pi\ell D}+\Gamma_b\delta_{\ell0}$ است. $|c_\ell|^2$ وزن طیفی پوش است. حساب توان مدولاسیون به تبادل پمپ، پاشندگی ماده، بهنجارسازی درگاه و همهٔ نوارهای جانبی نیاز دارد؛ مدل پوش آهسته آن حساب کامل را حل نمی‌کند.

In [ ]:
time=np.linspace(0,24e-6,301);bias=step_voltage(time);step=response(10.,bias,P)
u=(np.arange(8192)+.5)/8192;orders=np.arange(-10,11)
period_bias=periodic_voltage(u);slow=response(10.,period_bias,P)['gamma']
ga,gb=response(10.,[3.,14.],P)['gamma'];fast=binary_harmonics(orders,ga,gb)
slow_coeff=sampled_harmonics(slow,orders)
assert np.max(abs(sampled_harmonics(np.where(u<.5,ga,gb),orders)-fast))<1e-6
fig,axes=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
axes[0,0].plot(time*1e6,bias,color='#087f83');axes[0,0].axhline(14,color='#a6b6c4',ls=':');axes[0,0].axvline(-4*np.log(.02),color='#b36b10',ls='--');axes[0,0].set(xlabel='Time (μs)',ylabel='Bias (V)',title='2 → 14 V command; τ = 4 μs')
axes[0,1].plot(time*1e6,np.rad2deg(np.unwrap(np.angle(step['gamma']))),color='#087f83');axes[0,1].set(xlabel='Time (μs)',ylabel='Unwrapped reflection phase (degrees)',title='Phase follows the nonlinear circuit')
axes[1,0].plot(u,np.where(u<.5,3,14),color='#b36b10',ls='--',label='Command');axes[1,0].plot(u,period_bias,color='#087f83',label='Steady periodic RC');axes[1,0].set(xlabel='Cycle fraction t/T',ylabel='Bias (V)',title='20 kHz repeated switching');axes[1,0].legend()
axes[1,1].stem(orders-.12,abs(fast)**2,linefmt='C1-',markerfmt='C1o',basefmt=' ',label='Instantaneous binary');axes[1,1].stem(orders+.12,abs(slow_coeff)**2,linefmt='C0-',markerfmt='C0s',basefmt=' ',label='Finite RC time')
axes[1,1].set(xlabel='Sideband index ℓ: f = 10 GHz + ℓ × 20 kHz',ylabel='Envelope weight |cℓ|²',title='Spectral redistribution; not a full power budget');axes[1,1].legend(fontsize=8)
plt.show()
print('2% voltage settling time (μs):',-4*np.log(.02))
print('Full sampled-envelope Parseval residual:',abs(np.sum(abs(np.fft.ifft(slow))**2)-np.mean(abs(slow)**2)))

## ۶. ممیزی و ذخیرهٔ مرجع مرورگر

ممیزی مستقل، معادلات مرزی امپدانس را حل می‌کند، معادلهٔ دیفرانسیل RC را انتگرال می‌گیرد، انتگرال هارمونیک را با FFT و ضرایب دودویی تحلیلی مقایسه و ورودی نامعتبر را رد می‌کند. مرجع پیش‌فرض، آزمایش ویرایش‌شده را تأیید نمی‌کند. مرورگر از نتایج ذخیره‌شدهٔ همین مولد استفاده می‌کند و گزینهٔ دانلود آزمایشگاه مرورگری ندارد.

**تمرین:** کدنامهٔ ولتاژ یکنواخت را با ولتاژهای فازآگاه جایگزین کنید؛ اندازه‌گیری نویزی و تکرارپذیری را بیازمایید؛ قانون ظرفیت را تغییر و دسترس‌پذیری هدف را کنترل کنید؛ زمان نامعتبرشدن مدل حالت منجمد را مشخص کنید. سلول بایاس‌شدهٔ تمام‌موج، آرایهٔ دارای کوپلینگ و بازخورد سخت‌افزاری مراحل بعدی اعتبارسنجی‌اند.

In [ ]:
from scripts.validate_chapter07 import validate
audit=validate();assert audit['status']=='passed'
def curve(f):
    z=response(f,VOLTAGE,P)
    return dict(frequency_GHz=f,real=z['gamma'].real.tolist(),imag=z['gamma'].imag.tolist(),
                reflected=z['reflected'].tolist(),phase=np.rad2deg(np.unwrap(np.angle(z['gamma']))).tolist())
beam_cases=[]
for target_angle in range(-40,41,5):
    variants=[]
    for label,bits,circuit_flag in [('ideal1',1,False),('ideal2',2,False),('ideal4',4,False),('circuit4',4,True)]:
        d=code_array(float(target_angle),bits,circuit_flag)
        variants.append(dict(key=label,intensity=(abs(array_field(d['gamma'],ANGLES,d['positions']))**2).tolist(),
                             voltage=d['voltage_V'].tolist() if circuit_flag else None,indices=d['indices'].tolist(),target_metric=d['target_metric']))
    beam_cases.append(dict(target_deg=target_angle,variants=variants))
reference=dict(metadata=dict(schema='tunable-surface-v1',source_sha256=audit['source_sha256'],parameters=asdict(P),evidence='analytical/synthetic only'),
               voltage=VOLTAGE.tolist(),cell_curves=[curve(float(f)) for f in range(7,14)],angles=ANGLES.tolist(),beams=beam_cases,
               feedback=feedback,dynamics=dict(time_us=(time*1e6).tolist(),voltage=bias.tolist(),phase=np.rad2deg(np.unwrap(np.angle(step['gamma']))).tolist(),
               cycle=u[::16].tolist(),period_voltage=period_bias[::16].tolist(),orders=orders.tolist(),fast_weights=(abs(fast)**2).tolist(),slow_weights=(abs(slow_coeff)**2).tolist()))
path=ROOT/'data/chapter-07/reference.json';path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(json.dumps(reference,separators=(',',':'),allow_nan=False)+'\n',encoding='utf-8')
print(json.dumps(audit['checks'],indent=2));print('Saved data/chapter-07/reference.json')